# QuantJourney SDK - CBOE VIX Volatility Index

This notebook demonstrates CBOE data:
- VIX Fear Index (current and historical)
- Volatility analysis
- Fear/Greed zones

**API:** https://api.quantjourney.cloud

In [ ]:
import sys
sys.path.insert(0, '..')
from quantjourney.sdk import QuantJourneyAPI
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'png'
import plotly.graph_objects as go
import os
API_KEY = os.environ['QJ_API_KEY']
qj = QuantJourneyAPI(api_key=API_KEY)
print('✓ Connected to QuantJourney API')


## 1. VIX Historical Data (CBOE)

In [ ]:
from datetime import datetime, timedelta
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')
try:
    response = qj.cboe.get_vix_data(start_date=start_date, end_date=end_date, interval='1d')
    vix_data = response.get('value', response) if isinstance(response, dict) else response
    if vix_data:
        df_vix_cboe = pd.DataFrame(vix_data)
        print(f'VIX records from CBOE: {len(df_vix_cboe)}')
        print(f'\nColumns: {list(df_vix_cboe.columns)}')
        print(f'\nLatest VIX data:')
        print(df_vix_cboe.tail())
except Exception as e:
    print(f'Note: CBOE VIX data - {e}')


## 2. VIX Historical (via EOD)

In [ ]:
try:
    response = qj.eod.get_historical_prices(symbol='VIX.INDX', start_date='2020-01-01', end_date='2025-12-31', frequency='1d')
    vix_hist = response.get('value', response) if isinstance(response, dict) else response
    if vix_hist:
        df_vix = pd.DataFrame(vix_hist)
        df_vix['date'] = pd.to_datetime(df_vix['date'])
        df_vix = df_vix.sort_values('date')
        print(f'VIX records: {len(df_vix)}')
        print(f"Date range: {df_vix['date'].min().date()} to {df_vix['date'].max().date()}")
        print(f'\nLatest values:')
        print(df_vix[['date', 'open', 'high', 'low', 'close']].tail())
except Exception as e:
    print(f'Note: VIX historical - {e}')


In [ ]:
if 'df_vix' in dir() and len(df_vix) > 0:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df_vix['date'], y=df_vix['close'], fill='tozeroy', name='VIX', line=dict(color='orange')))
    fig.add_hline(y=20, line_dash='dash', line_color='green', annotation_text='Low Fear (<20)')
    fig.add_hline(y=30, line_dash='dash', line_color='red', annotation_text='High Fear (>30)')
    fig.add_hline(y=40, line_dash='dash', line_color='darkred', annotation_text='Extreme Fear (>40)')
    fig.update_layout(title='VIX - Fear Index (2020-Present)', yaxis_title='VIX Level', xaxis_title='Date', template='plotly_dark', height=500)
    fig.show()
    print(f'\nVIX Analysis:')
    print(f"  Current:   {df_vix['close'].iloc[-1]:.1f}")
    print(f"  Average:   {df_vix['close'].mean():.1f}")
    print(f"  Median:    {df_vix['close'].median():.1f}")
    print(f"  Max Spike: {df_vix['close'].max():.1f}")
    print(f"  Min:       {df_vix['close'].min():.1f}")
    current = df_vix['close'].iloc[-1]
    if current < 15:
        sentiment = 'Extreme Complacency'
    elif current < 20:
        sentiment = 'Low Fear'
    elif current < 30:
        sentiment = 'Moderate Fear'
    elif current < 40:
        sentiment = 'High Fear'
    else:
        sentiment = 'Extreme Fear/Panic'
    print(f'  Sentiment: {sentiment}')


In [ ]:
if 'df_vix' in dir() and len(df_vix) > 0:
    df_vix['vix_ma20'] = df_vix['close'].rolling(20).mean()
    df_vix['vix_ma50'] = df_vix['close'].rolling(50).mean()
    df_vix['vix_std20'] = df_vix['close'].rolling(20).std()
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df_vix['date'], y=df_vix['close'], name='VIX', line=dict(color='orange', width=1)))
    fig.add_trace(go.Scatter(x=df_vix['date'], y=df_vix['vix_ma20'], name='20-day MA', line=dict(color='cyan', width=2)))
    fig.add_trace(go.Scatter(x=df_vix['date'], y=df_vix['vix_ma50'], name='50-day MA', line=dict(color='magenta', width=2)))
    fig.update_layout(title='VIX with Moving Averages', yaxis_title='VIX Level', template='plotly_dark', height=450)
    fig.show()
    low_vol = (df_vix['close'] < 20).sum() / len(df_vix) * 100
    mid_vol = ((df_vix['close'] >= 20) & (df_vix['close'] < 30)).sum() / len(df_vix) * 100
    high_vol = (df_vix['close'] >= 30).sum() / len(df_vix) * 100
    print(f'\nVIX Regime Distribution:')
    print(f'  Low Vol (<20):     {low_vol:.1f}%')
    print(f'  Medium (20-30):    {mid_vol:.1f}%')
    print(f'  High Vol (>30):    {high_vol:.1f}%')


## Summary

CBOE VIX data covered:
- **Current VIX**: Real-time fear gauge
- **Historical VIX**: Long-term volatility trends
- **VIX Fear**: Treshold volatility plots
- **Regime Analysis**: Low/Medium/High volatility periods

### VIX Interpretation:
- **< 15**: Extreme complacency (potential top)
- **15-20**: Low fear, bullish sentiment
- **20-30**: Normal/elevated uncertainty
- **30-40**: High fear, potential opportunity
- **> 40**: Panic selling, capitulation